# Task 13: Temporal Convolutional Networks (TCN) with Dilated Causal Convolutions

**Objective:** Model sequential patterns and time-series history without temporal look-ahead leaks by implementing 1D causal convolutions featuring exponentially increasing dilation factors ($d=2^l$).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Custom layer to crop/slice future outputs from convolution
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
        
    def forward(self, x):
        # x shape: (B, C, L)
        # Slice out future indices introduced by symmetric padding
        return x[:, :, :-self.chomp_size].contiguous()

# 1. Dilated Causal Residual Block
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, dilation, padding, dropout=0.2):
        super().__init__()
        # First dilated causal Conv
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        
        # Second dilated causal Conv
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        
        self.net = nn.Sequential(
            self.conv1, self.chomp1, self.relu1, self.dropout1,
            self.conv2, self.chomp2, self.relu2, self.dropout2
        )
        
        # Linear projection shortcut if input/output dimensions differ
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.relu = nn.ReLU()
        
    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

In [ ]:
# Assemble multi-layered TCN and test with a 1D sequence
class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=2, dropout=0.2):
        super().__init__()
        layers = []
        num_levels = len(num_channels)
        
        for i in range(num_levels):
            # Systematically double the dilation factor: 1, 2, 4, 8...
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            
            # To ensure output matches input length, padding must equal (kernel_size - 1) * dilation
            padding_size = (kernel_size - 1) * dilation_size
            
            layers.append(
                TemporalBlock(in_channels, out_channels, kernel_size, stride=1, 
                              dilation=dilation_size, padding=padding_size, dropout=dropout)
            )
            
        self.network = nn.Sequential(*layers)
        
    def forward(self, x): return self.network(x)

# Test sequence shape (B, C, L) where length = 100
B, C, L = 4, 1, 100
x = torch.randn(B, C, L)

tcn = TemporalConvNet(num_inputs=1, num_channels=[8, 16, 32], kernel_size=3)
out = tcn(x)

print("Input Sequence Shape: ", x.shape)
print("Output Sequence Shape:", out.shape)
assert out.shape == (B, 32, L), "Verification failed: output length modified!"
print("Success: TCN causal convolutions successfully compiled!")